# Day 3, hands-on 1: the full pass

Profile the file, decide per field, clean it, and reconcile the counts, with a check after every step.

Every decision in this notebook is a lettered choice written in a comment above a `__TODOn__`
placeholder. Replace each placeholder with the option you pick, run the cell, and read the check
underneath it. The checks are how you know you are right without opening the solution.

Run this notebook as it stands and it stops at the first placeholder with a `NameError` naming
`__TODO1__`. That is the file working correctly, so do not report it as a bug.

Where this sits in the day, and the steps this notebook walks.

In [ ]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["profiling the columns", "the rows a profile cannot see", "hands-on: the full pass", "hands-on: which dataset"], lit=2, title="the day's notebooks", show=False),
    kit.flow(["profile every field", "decide per field", "clean and reject", "reconcile", "the decisions log"], title="this notebook's steps", show=False),
)

## Setup

The fifty Kalpa Retail orders at their dirtiest, straight from `../data/`.

In [ ]:
orders = kit.load_csv("C2_W01_D03_orders_STUDENT.csv")

print(len(orders), "rows read from ../data/")
print(list(orders[0]))

## Step 1. Profile every field

Three counts per field, before you touch anything. The third count is the one that can warn you later.

In [ ]:
kit.flow(["profile every field", "decide per field", "clean and reject", "reconcile", "the decisions log"], lit=0)

In [ ]:
# TODO 1. How do you count the different values a field holds?
#   a) len(rows)
#   b) len({r[field] for r in rows})
#   c) len([r[field] for r in rows])
#   d) sum(1 for r in rows if r[field])
def profile(rows, field):
    present = sum(1 for r in rows if r[field].strip())
    converts = sum(1 for r in rows if r[field].strip().lstrip("-").isdigit())
    distinct = __TODO1__
    return {"present": present, "converts": converts, "distinct": distinct}

amount = profile(orders, "amount")
print(amount)

In [ ]:
kit.check("amount is present on 48 rows", amount["present"] == 48, str(amount))
kit.check("44 of them convert", amount["converts"] == 44)
kit.check("46 distinct values", amount["distinct"] == 46)

## Step 2. Read the shape rather than the numbers

A profile is only useful if you can say what kind of field each column is. Do that for order_id, which is the field the next notebook turns on.

In [ ]:
kit.flow(["profile every field", "decide per field", "clean and reject", "reconcile", "the decisions log"], lit=1)

In [ ]:
# TODO 2. What does 49 distinct order_ids across 50 rows say?
#   a) the field is a category
#   b) one row is empty
#   c) something repeats that should not
#   d) the file is sorted
oid = profile(orders, "order_id")
seg = profile(orders, "segment")

reading = __TODO2__
print(oid, seg)
print(reading)

In [ ]:
kit.check("order_id has 49 distinct values across 50 rows", oid["distinct"] == 49)
kit.check("segment is a category with four values", seg["distinct"] == 4)
kit.check("your reading names the repeat", "repeat" in reading)

## Step 3. Decide per field, and write the reason down

Two fields are incomplete and they get opposite treatment. The decision is the deliverable; the code is the easy part.

In [ ]:
kit.flow(["profile every field", "decide per field", "clean and reject", "reconcile", "the decisions log"], lit=2)

In [ ]:
# TODO 3. What do you do with a field whose absence means something?
#   a) fill it with zero
#   b) drop those rows
#   c) fill it with the average
#   d) keep absent as absent, and flag it
decisions = []

def record(field, finding, choice, reason):
    decisions.append({"field": field, "finding": finding, "choice": choice, "reason": reason})

record("amount", "6 of 50 will not convert", "reject the row", "an order with no usable amount cannot be summed")
record("discount", "absent on 39 of 50", __TODO3__,
       "absence means no discount ran, which is a fact rather than a gap")

print(len(decisions), "decisions recorded")
for d in decisions:
    print(" ", d["field"], "->", d["choice"])

In [ ]:
kit.check("two decisions are recorded", len(decisions) == 2)
kit.check("each one carries a reason a reviewer could argue with",
          all(d["reason"] for d in decisions))
kit.check("the discount decision keeps the absence", "absent" in decisions[1]["choice"])

## Step 4. Clean, reject, and reconcile

The pass itself. What matters is not the clean file; it is that the three counts add up.

In [ ]:
kit.flow(["profile every field", "decide per field", "clean and reject", "reconcile", "the decisions log"], lit=3)

In [ ]:
# TODO 4. Which expression is the reconciliation?
#   a) len(clean) > len(rejects)
#   b) len(clean) == 44
#   c) len(clean) + len(rejects) == len(orders)
#   d) len(orders) == 50
clean, rejects = [], []
for r in orders:
    raw = r["amount"].strip()
    if raw.lstrip("-").isdigit():
        clean.append({**r, "amount": int(raw)})
    else:
        rejects.append({"order_id": r["order_id"], "reason": f"amount will not convert: {raw!r}"})

holds = __TODO4__
print(len(orders), len(clean), len(rejects), holds)

In [ ]:
kit.check("44 rows survive", len(clean) == 44, f"{len(clean)} clean")
kit.check("6 are rejected", len(rejects) == 6)
kit.check("the reconciliation holds", holds is True)

## Step 5. What ships

Two files and a log. Somebody who was not here has to be able to reconstruct your count from them.

In [ ]:
kit.flow(["profile every field", "decide per field", "clean and reject", "reconcile", "the decisions log"], lit=4)

In [ ]:
# TODO 5. What leaves this notebook?
#   a) the clean file only
#   b) the clean file and the rejects file
#   c) the clean file, the rejects file and the decisions log
#   d) the notebook itself
ships = __TODO5__
print(ships)
print("rejected ids:", [r["order_id"] for r in rejects])

In [ ]:
kit.check("three artifacts ship", len(ships) == 3)
kit.check("the decisions log is one of them", "decisions log" in ships)
kit.check("all six rejected ids are named",
          len([r["order_id"] for r in rejects]) == 6)

## What to post

Post one line with the five letters, then the reconciliation:

```
1b 2c 3d 4c 5c
50 in = 44 clean + 6 rejected
```

Then one sentence defending your discount decision to somebody who wanted it filled with zero.

In [ ]:
kit.flow(["profile every field", "decide per field", "clean and reject", "reconcile", "the decisions log"], lit=4, title="the notebook, end to end")
kit.check_summary()